# Fine-tune Decoder-Only Transformer on Tiny Shakespeare

This notebook loads a previously trained `DecoderOnlyTransformer` from `model_weights.pth` and continues training (fine-tuning) on the same `tiny_corpus.txt` dataset with a different learning rate.

In [1]:
import torch
from Transformer import DecoderOnlyTransformer
torch.manual_seed(1337)

if torch.xpu.is_available():
    device = torch.device("xpu")
    print(f"XPU detected: {torch.xpu.get_device_properties(0).name}")
else:
    device = torch.device("cpu")
    print("XPU not available. Using CPU.")

XPU detected: Intel(R) Arc(TM) A770 Graphics


In [ ]:
batch_size = 64
block_size = 256

n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

ft_max_iters = 500 
ft_eval_interval = 100
ft_learning_rate = 3e-4
ft_eval_iters = 200

In [ ]:
with open('tiny_shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print('Vocab size:', vocab_size)

Vocab size: 65


In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
print('Data shape:', data.shape)

def split_data(data, split=0.9):
    n = int(split * len(data))
    return data[:n], data[n:]

train_data, val_data = split_data(data)
print(f'Train data length: {len(train_data)}')
print(f'Val data length: {len(val_data)}')

Data shape: torch.Size([1115394])
Train data length: 1003854
Val data length: 111540


In [114]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('Batch shapes:', x.shape, y.shape)

Batch shapes: torch.Size([64, 256]) torch.Size([64, 256])


In [ ]:
model = DecoderOnlyTransformer(
    vocab_size=vocab_size,
    embed_dim=n_embd,
    n_head=n_head,
    n_layer=n_layer,
    max_seq_len=block_size,
    dropout=dropout,
).to(device)

checkpoint = torch.load('model_weights_finetuned.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print('Loaded weights from model_weights_finetuned.pth')

print('Model parameters (M):', sum(p.numel() for p in model.parameters()) / 1e6)

Loaded weights from model_weights_finetuned.pth
Model parameters (M): 10.788929


In [121]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(ft_eval_iters)
        for k in range(ft_eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

losses = estimate_loss()
print(f'train loss {losses["train"]:.4f}, val loss {losses["val"]:.4f}')

train loss 1.1793, val loss 1.4838


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=ft_learning_rate)

In [ ]:
for iter in range(ft_max_iters):

    if iter % ft_eval_interval == 0 or iter == ft_max_iters - 1:
        losses = estimate_loss()
        print(f'step {iter}: train loss {losses["train"]:.4f}, val loss {losses["val"]:.4f}')

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 1.1784, val loss 1.4907
step 100: train loss 1.1782, val loss 1.4918
step 199: train loss 1.1772, val loss 1.4910


In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=1000, block_size=block_size)[0].tolist()))


A dares instant-scorn! hath remember ere thou,
And Duke of York, his guilty divines,
Cut up my mother most have add life.
So perland awake leave for argument;
Who, that never governments me reposed.

RICHMOND:
Why letter, then have lost to me my bed
Of the unking of our sorrow-lie-like heart no time
By my lord fortune plain-flatterer'd fools;
And now I'll look like before him in their
As the warls; our eyes, here comes the earth,
On thou wilt laid them fault, like a wife.

NORTHUMBERLAND:
Tut, no put, no, good not shall give, still thy
Ere of your great gratisformity.

FRIAR LAURENCE:
My lords Hastings, sir? fray, where our good ere 's name:
Blessed his husband,'
This way his forends shall we from my trongs, he
most trembling wife to her? his fires about womb;
Which, we have the king, and men that pervice,
I'll not be kindness.

DORCAS:
Nay, sir, you but it on reason
That to give us it.

CORIOLANUS:
Why, my Barnardine?

CORIOLANUS:
Why, then you are to once I as he?

ESCALUS:
Marry, y

In [ ]:
save_weights = {
    'model_state_dict': model.state_dict(),
}
torch.save(save_weights, 'model_weights_finetuned.pth')
